`233 countries` were there in the original LPI csv file, after extracting the countries with `full indicator`, we left with
`109 countries` and it had `regional aggregate`, `income group`, etc. 

>Remove these 13 entries

>Code	Label	What it represents\
> EAS	East Asia & Pacific	    `Regional aggregate`\
> ECS	Europe & Central Asia	`Regional aggregate`\
> G_COA	Coastal Countries	    `Special country group`\
> G_SIC	Small Island Countries       `Special country group`\
> HIC	High income	`Income group`\
> LCN	Latin America & Caribbean	`Regional aggregate`\
> LIC	Low income	`Income group`\
> LMC	Lower middle income	`Income group`\
> MEA	Middle East, North Africa, Afghanistan & Pakistan	`Regional aggregate`\
> NAC	North America	`Regional aggregate`\
> SAS	South Asia	`Regional aggregate`\
> SSF	Sub-Saharan Africa	`Regional aggregate`\
> UMC	Upper middle income	`Income group`


so ultimately left with `96 countries`.

In [1]:
import pandas as pd
import numpy as np


In [15]:
dflp=pd.read_csv("WB_LPI_20_WIDEF.csv")
dflp.head()

,REF_AREA,INDICATOR,METRIC,UNIT_MEASURE,REF_AREA_LABEL,INDICATOR_LABEL,METRIC_LABEL,UNIT_MEASURE_LABEL,2024
0,NZL,PC_B2B,_Z,PRTN,New Zealand,Number of business-to-business postal partner ...,Not Applicable,Partners,108.5
1,AIA,LT_M,M,D,Anguilla,Lead time to import from port of origin to des...,Median,Days,44.5
2,GAB,PO_B2C,IQR_80_20,D,Gabon,Business-to-consumer postal delivery time,Interquartile Range (P80-P20),Days,51.1
3,COD,AV_DT,M,D,"Congo, Dem. Rep.",Aviation import dwell time,Median,Days,3.7
4,TTO,SC_START,MD,D,Trinidad and Tobago,Export supply chain initiation,Median,Days,6.0


In [ ]:
dflp["REF_AREA"].nunique()
# 238 countries are there, that what this means. . . . . .

238

In [17]:
# Inspect the dataset
print("Dataset dimensions:", dflp.shape)
print("\nColumn names:")
print(dflp.columns.tolist())

print("\nFirst five rows:")
display(dflp.head())

print("\nData types:")
display(dflp.dtypes.to_frame("Data type"))


Dataset dimensions: (8115, 9)

Column names:
['REF_AREA', 'INDICATOR', 'METRIC', 'UNIT_MEASURE', 'REF_AREA_LABEL', 'INDICATOR_LABEL', 'METRIC_LABEL', 'UNIT_MEASURE_LABEL', '2024']

First five rows:


,REF_AREA,INDICATOR,METRIC,UNIT_MEASURE,REF_AREA_LABEL,INDICATOR_LABEL,METRIC_LABEL,UNIT_MEASURE_LABEL,2024
0,NZL,PC_B2B,_Z,PRTN,New Zealand,Number of business-to-business postal partner ...,Not Applicable,Partners,108.5
1,AIA,LT_M,M,D,Anguilla,Lead time to import from port of origin to des...,Median,Days,44.5
2,GAB,PO_B2C,IQR_80_20,D,Gabon,Business-to-consumer postal delivery time,Interquartile Range (P80-P20),Days,51.1
3,COD,AV_DT,M,D,"Congo, Dem. Rep.",Aviation import dwell time,Median,Days,3.7
4,TTO,SC_START,MD,D,Trinidad and Tobago,Export supply chain initiation,Median,Days,6.0



Data types:


,Data type
REF_AREA,str
INDICATOR,str
METRIC,str
UNIT_MEASURE,str
REF_AREA_LABEL,str
INDICATOR_LABEL,str
METRIC_LABEL,str
UNIT_MEASURE_LABEL,str
2024,float64


In [18]:

# 1. Select the six required indicators and their metrics
selected_metrics = {
    "CT_MC": "_Z",    # Maritime connectivity
    "CT_DT_M": "M",   # Container dwell time — mean (as assumed)
    "AV_PRTN": "_Z",  # Aviation connectivity
    "AV_DT": "M",     # Aviation dwell time — mean (as assumed)
    "PC_B2B": "_Z",   # B2B postal connectivity
    "PO_B2B": "M"     # B2B postal delivery time — mean (as assumed)
}

indicator_names = {
    "CT_MC": "Maritime connectivity",
    "CT_DT_M": "Container dwell time",
    "AV_PRTN": "Aviation connectivity",
    "AV_DT": "Aviation dwell time",
    "PC_B2B": "B2B postal connectivity",
    "PO_B2B": "B2B postal delivery time"
}

# 2. Keep only the selected indicator-metric combinations
parts = []

for indicator, metric in selected_metrics.items():
    part = dflp[
        (dflp["INDICATOR"] == indicator) &
        (dflp["METRIC"] == metric)
    ].copy()

    parts.append(part)

selected = pd.concat(parts, ignore_index=True)

# 3. Reshape to one row per economy
selected["Criterion"] = selected["INDICATOR"].map(indicator_names)

wide = selected.pivot(
    index=["REF_AREA", "REF_AREA_LABEL"],
    columns="Criterion",
    values="2024"
).reset_index()

wide.columns.name = None

criteria = list(indicator_names.values())

# 4. Retain economies with all six non-missing observations
missing_mask = wide[criteria].isna().any(axis=1)

excluded = wide.loc[missing_mask].copy()
complete = wide.loc[~missing_mask].copy()

# 5. Document which criteria were missing for each excluded economy
excluded["Missing indicators"] = excluded.apply(
    lambda row: ", ".join(
        criterion for criterion in criteria
        if pd.isna(row[criterion])
    ),
    axis=1
)

# 6. Print results
print("Economies before completeness filter:", len(wide))
print("Economies retained:", len(complete))
print("Economies excluded:", len(excluded))

print("\nMissing values by criterion:")
display(wide[criteria].isna().sum().to_frame("Missing count"))

print("\nExcluded economies and missing fields:")
display(
    excluded[
        ["REF_AREA", "REF_AREA_LABEL", "Missing indicators"]
    ].sort_values("REF_AREA_LABEL")
)

print("\nFinal complete dataset:")
display(complete.head(10))

# 7. Save both files
complete.to_csv("LPI_2024_complete.csv", index=False)
excluded[
    ["REF_AREA", "REF_AREA_LABEL", "Missing indicators"]
].to_csv("LPI_2024_exclusions.csv", index=False)

print("\nSaved: LPI_2024_complete.csv")
print("Saved: LPI_2024_exclusions.csv")

# Final check
assert complete[criteria].notna().all().all()
print("SUCCESS: Every retained economy has all six observations.")


Economies before completeness filter: 233
Economies retained: 109
Economies excluded: 124

Missing values by criterion:


,Missing count
Maritime connectivity,51
Container dwell time,57
Aviation connectivity,98
Aviation dwell time,98
B2B postal connectivity,27
B2B postal delivery time,32



Excluded economies and missing fields:


,REF_AREA,REF_AREA_LABEL,Missing indicators
1,AFG,Afghanistan,"Maritime connectivity, Container dwell time, A..."
4,ALB,Albania,"Aviation connectivity, Aviation dwell time"
8,ASM,American Samoa,"Aviation connectivity, Aviation dwell time, B2..."
3,AIA,Anguilla,"Maritime connectivity, Aviation connectivity, ..."
7,ARM,Armenia,"Maritime connectivity, Container dwell time, A..."
...,...,...,...
226,WLF,Wallis-et-Futuna (Fr.),"Aviation connectivity, Aviation dwell time"
174,PSE,West Bank and Gaza,"Maritime connectivity, Container dwell time, A..."
229,YEM,"Yemen, Rep.","Aviation connectivity, Aviation dwell time, B2..."
231,ZMB,Zambia,"Maritime connectivity, Container dwell time"



Final complete dataset:


,REF_AREA,REF_AREA_LABEL,Aviation connectivity,Aviation dwell time,B2B postal connectivity,B2B postal delivery time,Container dwell time,Maritime connectivity
0,ABW,Aruba,35.0,2.6,34.5,4.9,7.1,12.0
2,AGO,Angola,57.5,8.0,44.5,15.6,7.8,28.0
5,ARE,United Arab Emirates,142.0,1.4,144.0,2.9,5.5,61.0
6,ARG,Argentina,104.5,2.3,84.0,22.4,8.0,23.0
9,ATG,Antigua and Barbuda,30.0,2.1,23.5,14.4,6.1,23.0
10,AUS,Australia,114.5,0.8,119.0,3.3,2.2,41.0
14,BEL,Belgium,148.0,0.6,153.0,5.0,4.8,88.0
18,BGD,Bangladesh,81.5,4.3,55.5,16.9,8.3,13.0
20,BHR,Bahrain,83.0,2.0,66.5,3.8,6.7,14.0
21,BHS,"Bahamas, The",29.0,3.9,19.0,26.1,9.8,44.0



Saved: LPI_2024_complete.csv
Saved: LPI_2024_exclusions.csv
SUCCESS: Every retained economy has all six observations.


In [20]:
print(sorted(dflp["REF_AREA_LABEL"].unique()))

['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas, The', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Bonaire (Neth.)', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'British Virgin Islands', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China', 'Coastal Countries', 'Colombia', 'Comoros', 'Congo, Dem. Rep.', 'Congo, Rep.', 'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czechia', 'Côte d’Ivoire', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'East Asia & Pacific', 'Ecuador', 'Egypt, Arab Rep.', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Europe & Central Asia', 'Faroe Islands', 'Fiji', 'Finland'

In [23]:
df_new=pd.read_csv("LPI_final.csv")
df_new.describe()

,Aviation connectivity,Aviation dwell time,B2B postal connectivity,B2B postal delivery time,Maritime connectivity,Container dwell time
count,96.000000,96.000000,96.000000,96.000000,96.00000,96.000000
mean,96.046875,2.740625,82.911458,9.440625,38.15625,7.079167
std,37.429940,1.929689,41.197023,8.909269,24.42403,4.249208
min,20.000000,0.300000,13.500000,1.400000,6.00000,2.100000
25%,71.625000,1.475000,55.375000,4.175000,18.00000,4.575000
50%,95.250000,2.150000,77.250000,6.100000,33.00000,6.100000
75%,124.000000,3.225000,109.000000,13.225000,52.25000,8.000000
max,167.000000,12.100000,190.000000,57.700000,97.00000,23.600000


`ok all stats still correct even after removing those 15 countries . . `

# Reference Ranking

- `min max normalizing`
- `reversing three indicator` to match the higher = better.

In [2]:
import pandas as pd
df_r= pd.read_csv("LPI_final.csv")
df_r.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   REF_AREA                  96 non-null     str    
 1   REF_AREA_LABEL            96 non-null     str    
 2   Aviation connectivity     96 non-null     float64
 3   Aviation dwell time       96 non-null     float64
 4   B2B postal connectivity   96 non-null     float64
 5   B2B postal delivery time  96 non-null     float64
 6   Maritime connectivity     96 non-null     int64  
 7   Container dwell time      96 non-null     float64
dtypes: float64(5), int64(1), str(2)
memory usage: 7.2 KB


In [3]:
df_r.head()

,REF_AREA,REF_AREA_LABEL,Aviation connectivity,Aviation dwell time,B2B postal connectivity,B2B postal delivery time,Maritime connectivity,Container dwell time
0,ABW,Aruba,35.0,2.6,34.5,4.9,12,7.1
1,AGO,Angola,57.5,8.0,44.5,15.6,28,7.8
2,ARE,United Arab Emirates,142.0,1.4,144.0,2.9,61,5.5
3,ARG,Argentina,104.5,2.3,84.0,22.4,23,8.0
4,ATG,Antigua and Barbuda,30.0,2.1,23.5,14.4,23,6.1


In [4]:
df_r.describe()

,Aviation connectivity,Aviation dwell time,B2B postal connectivity,B2B postal delivery time,Maritime connectivity,Container dwell time
count,96.000000,96.000000,96.000000,96.000000,96.00000,96.000000
mean,96.046875,2.740625,82.911458,9.440625,38.15625,7.079167
std,37.429940,1.929689,41.197023,8.909269,24.42403,4.249208
min,20.000000,0.300000,13.500000,1.400000,6.00000,2.100000
25%,71.625000,1.475000,55.375000,4.175000,18.00000,4.575000
50%,95.250000,2.150000,77.250000,6.100000,33.00000,6.100000
75%,124.000000,3.225000,109.000000,13.225000,52.25000,8.000000
max,167.000000,12.100000,190.000000,57.700000,97.00000,23.600000


In [5]:

benefit_criteria = [
    "Aviation connectivity",
    "B2B postal connectivity",
    "Maritime connectivity"
]

cost_criteria = [
    "Aviation dwell time",
    "B2B postal delivery time",
    "Container dwell time"
]

dflp=df_r.copy()
normalised = dflp.copy()

# Benefit criteria: higher values are better
for col in benefit_criteria:
    normalised[col] = (
        (dflp[col] - dflp[col].min())
        / (dflp[col].max() - dflp[col].min())
    )

# Cost criteria: lower values are better
for col in cost_criteria:
    normalised[col] = (
        (dflp[col].max() - dflp[col])
        / (dflp[col].max() - dflp[col].min())
    )

print(normalised.head())


  REF_AREA        REF_AREA_LABEL  Aviation connectivity  Aviation dwell time  \
0      ABW                 Aruba               0.102041             0.805085   
1      AGO                Angola               0.255102             0.347458   
2      ARE  United Arab Emirates               0.829932             0.906780   
3      ARG             Argentina               0.574830             0.830508   
4      ATG   Antigua and Barbuda               0.068027             0.847458   

   B2B postal connectivity  B2B postal delivery time  Maritime connectivity  \
0                 0.118980                  0.937833               0.065934   
1                 0.175637                  0.747780               0.241758   
2                 0.739377                  0.973357               0.604396   
3                 0.399433                  0.626998               0.186813   
4                 0.056657                  0.769094               0.186813   

   Container dwell time  
0              0.7

In [6]:
normalised.head()

,REF_AREA,REF_AREA_LABEL,Aviation connectivity,Aviation dwell time,B2B postal connectivity,B2B postal delivery time,Maritime connectivity,Container dwell time
0,ABW,Aruba,0.102041,0.805085,0.118980,0.937833,0.065934,0.767442
1,AGO,Angola,0.255102,0.347458,0.175637,0.747780,0.241758,0.734884
2,ARE,United Arab Emirates,0.829932,0.906780,0.739377,0.973357,0.604396,0.841860
3,ARG,Argentina,0.574830,0.830508,0.399433,0.626998,0.186813,0.725581
4,ATG,Antigua and Barbuda,0.068027,0.847458,0.056657,0.769094,0.186813,0.813953


In [7]:

criteria = benefit_criteria + cost_criteria

# Equal-weight average of the six normalised criteria
normalised["Reference score"] = normalised[criteria].mean(axis=1)

# Sort from highest score (best) to lowest score
reference_ranking = normalised.sort_values(
    by="Reference score",
    ascending=False
).reset_index(drop=True)

# Assign ranks starting from 1
reference_ranking["Rank"] = range(1, len(reference_ranking) + 1)

print(reference_ranking[
    ["Rank", "REF_AREA", "REF_AREA_LABEL", "Reference score"]
].head(10))

reference_ranking.to_csv("reference_ranking.csv", index=False)
print("its done. . . ")

   Rank REF_AREA  REF_AREA_LABEL  Reference score
0     1      USA   United States         0.963244
1     2      GBR  United Kingdom         0.944202
2     3      NLD     Netherlands         0.924940
3     4      FRA          France         0.909808
4     5      DEU         Germany         0.894230
5     6      BEL         Belgium         0.891211
6     7      SGP       Singapore         0.882732
7     8      ESP           Spain         0.875788
8     9      CHN           China         0.866343
9    10      ITA           Italy         0.842808
its done. . . 
